# ScaleFlow — AI/ML Initial Modeling Setup

**Enterprise Project Intelligence Platform**

This notebook prepares the initial AI/ML modeling structure for ScaleFlow. It
is built strictly from two source documents:

- **ScaleFlow — Project Analysis & Proposal** (BinX Tech, 2026)
- **ScaleFlow — AI/ML Model Approach** (BinX Tech, Team 4, 14 September 2026)

It sets up, in one place, the project structure, the environment, the
input/target definitions, the model pipelines, the preprocessing steps, and
reusable training/evaluation functions for the four ML problems defined in
the Model Approach document:

1. **Task Delay Prediction** — Binary Classification — Random Forest Classifier
2. **Risk Analysis** — Classification — Random Forest Classifier
3. **Bottleneck Detection** — Anomaly Detection — Isolation Forest
4. **Project Performance / Health Analysis** — Regression — Random Forest
   Regressor (**Phase 2**, deferred until a real numerical health target exists)

Per the Model Approach document (Section 11), the **Phase 1 / MVP** scope is
Task Delay Prediction, Risk Analysis, and Bottleneck Detection.

This setup does not train on real data yet — it is ready to receive the
prepared dataset from the data preparation task and run end-to-end as soon
as that dataset is provided.

## 1. Project Structure

The equivalent standalone project is organized as:

```
scaleflow_ml/
├── config.py                       # Global paths, random seed, task constants
├── main.py                         # Entry point to run the pipelines
├── requirements.txt                 # Python libraries needed
├── data/
│   ├── raw/                        # Place the prepared dataset here
│   └── processed/                  # Reserved for intermediate processed data
├── artifacts/
│   ├── models/                     # Trained model files (.joblib) are saved here
│   └── reports/                    # Evaluation reports (JSON + text) are saved here
├── notebooks/                      # This notebook
└── src/
    ├── features.py                  # Input/target schema for each ML problem
    ├── preprocessing.py             # Data loading, leakage guard, preprocessing
    ├── training.py                  # Reusable training / CV / tuning / saving
    ├── evaluation.py                # Reusable evaluation + reporting functions
    ├── pipeline.py                   # Orchestrates each task end-to-end
    └── models/
        ├── delay_prediction.py       # Task Delay Prediction — Random Forest
        ├── risk_analysis.py          # Risk Analysis — Random Forest
        ├── bottleneck_detection.py   # Bottleneck Detection — Isolation Forest
        └── performance_analysis.py   # Performance Analysis — Random Forest (Phase 2)
```

The cells below join every one of those files into a single runnable
notebook, in the same order the pipeline actually needs them: config →
features → preprocessing → models → training/evaluation utilities →
orchestration → run.

In [ ]:
# Set up the working folders so the notebook mirrors the standalone project
import os

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
MODELS_DIR = os.path.join(ARTIFACTS_DIR, "models")
REPORTS_DIR = os.path.join(ARTIFACTS_DIR, "reports")

for folder in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, REPORTS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Project folders ready:")
for folder in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, REPORTS_DIR]:
    print(" -", folder)

## 2. Required Python Libraries & Modeling Environment

The library list is scoped to what the Model Approach document actually
recommends: tree-based, interpretable models (Random Forest, Isolation
Forest), which live entirely inside `scikit-learn`.

| Library | Purpose |
|---|---|
| `pandas`, `numpy` | Data loading and manipulation |
| `scikit-learn` | Preprocessing, Random Forest Classifier/Regressor, Isolation Forest, evaluation metrics |
| `joblib` | Saving and loading trained models |
| `matplotlib`, `seaborn` | Visualization for EDA and evaluation |

Install (if needed) with:
```bash
pip install -r requirements.txt
```

In [ ]:
# Core imports used across every task in this notebook
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, KFold,
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    mean_absolute_error, root_mean_squared_error, r2_score,
)

print("Environment ready.")

## 3. Global Configuration

Central place for paths, the random seed, and task identifiers, shared by
all four ML problems (`config.py` in the standalone project).

In [ ]:
# Reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

# Task identifiers, used for saving/loading models and reports
TASK_DELAY_PREDICTION = "delay_prediction"
TASK_RISK_ANALYSIS = "risk_analysis"
TASK_BOTTLENECK_DETECTION = "bottleneck_detection"
TASK_PERFORMANCE_ANALYSIS = "performance_analysis"  # Phase 2 (deferred)

# Phase 1 = MVP models that can be built now (Model Approach, Section 11)
MVP_TASKS = [TASK_DELAY_PREDICTION, TASK_RISK_ANALYSIS, TASK_BOTTLENECK_DETECTION]
ALL_TASKS = MVP_TASKS + [TASK_PERFORMANCE_ANALYSIS]

# Expected proportion of tasks flagged as bottleneck/anomalous by Isolation Forest
BOTTLENECK_CONTAMINATION = 0.1

# Expected dataset filename once delivered by the data preparation task
RAW_DATASET_FILE = os.path.join(RAW_DATA_DIR, "scaleflow_tasks_dataset.csv")

## 4. Input & Target Definitions per ML Problem

Taken directly from Sections 4–7 of the Model Approach document. This is the
contract between the data preparation task and the modeling code below — if
a column name changes once the real dataset arrives, it only needs to be
updated here.

### 4.1 Task Delay Prediction (Section 4)
- **Problem type:** Binary classification
- **Target:** `is_delayed` — 1 if the actual finish date is after the
  planned due date, 0 otherwise
- **Numeric inputs:** progress percentage, estimated duration, days until
  deadline, task age, dependency count, delayed dependency count, workload
  ratio, active tasks count, historical delay rate, average completion time
- **Categorical inputs:** task status, priority, blocked status

### 4.2 Risk Analysis (Section 5)
- **Problem type:** Classification (Low / Medium / High), with a regression
  alternative if the dataset instead provides a continuous `risk_score`
- **Inputs, grouped exactly as in the document:**
  Schedule (delay probability, overdue task ratio, milestone status),
  Progress (project progress, progress variance),
  Dependencies (blocked tasks, delayed dependency count),
  Resources (workload ratio, team size, overloaded members),
  History (historical project delay rate, previous risk outcomes)

### 4.3 Bottleneck Detection (Section 6)
- **Problem type:** Anomaly detection (unsupervised — no labeled data exists yet)
- **Inputs:** task duration, completion velocity, blocked duration,
  dependency count, delayed dependency count, workload ratio, number of
  dependent tasks, historical completion time

### 4.4 Project Performance / Health Analysis (Section 7) — Phase 2
- **Problem type:** Regression
- **Target:** `health_score` — must not be fabricated; wait for a real
  numerical target
- **Inputs:** project progress percentage, overdue/completed/blocked task
  counts, workload ratio, team size, historical delay rate, progress
  variance, completion velocity, milestone status

In [ ]:
# ---- 4.1 Task Delay Prediction ----
DELAY_NUMERIC_FEATURES = [
    "progress_percentage", "estimated_duration", "days_until_deadline",
    "task_age", "dependency_count", "delayed_dependency_count",
    "workload_ratio", "active_tasks_count", "historical_delay_rate",
    "average_completion_time",
]
DELAY_CATEGORICAL_FEATURES = ["task_status", "priority", "blocked_status"]
DELAY_TARGET = "is_delayed"  # 1 = delayed, 0 = on-time
DELAY_FEATURES = DELAY_NUMERIC_FEATURES + DELAY_CATEGORICAL_FEATURES

# Fields that must NEVER be used as input features (Section 10: Avoiding Data
# Leakage) — they only exist to build the label after a task is finished.
DELAY_LEAKAGE_FIELDS = ["completed_date", "actual_duration", "final_delay_days"]

# ---- 4.2 Risk Analysis (grouped exactly as in the Model Approach doc) ----
RISK_SCHEDULE_FEATURES = ["delay_probability", "overdue_task_ratio"]
RISK_SCHEDULE_CATEGORICAL = ["milestone_status"]
RISK_PROGRESS_FEATURES = ["project_progress", "progress_variance"]
RISK_DEPENDENCY_FEATURES = ["blocked_tasks", "delayed_dependency_count"]
RISK_RESOURCE_FEATURES = ["workload_ratio", "team_size", "overloaded_members"]
RISK_HISTORY_FEATURES = ["historical_project_delay_rate"]
RISK_HISTORY_CATEGORICAL = ["previous_risk_outcomes"]

RISK_NUMERIC_FEATURES = (
    RISK_SCHEDULE_FEATURES + RISK_PROGRESS_FEATURES
    + RISK_DEPENDENCY_FEATURES + RISK_RESOURCE_FEATURES + RISK_HISTORY_FEATURES
)
RISK_CATEGORICAL_FEATURES = RISK_SCHEDULE_CATEGORICAL + RISK_HISTORY_CATEGORICAL
RISK_TARGET = "risk_level"                 # "low" / "medium" / "high"
RISK_TARGET_REGRESSION = "risk_score"       # alternative continuous target (0-100)
RISK_FEATURES = RISK_NUMERIC_FEATURES + RISK_CATEGORICAL_FEATURES

# ---- 4.3 Bottleneck Detection (unsupervised, no target column) ----
BOTTLENECK_NUMERIC_FEATURES = [
    "task_duration", "completion_velocity", "blocked_duration",
    "dependency_count", "delayed_dependency_count", "workload_ratio",
    "num_dependent_tasks", "historical_completion_time",
]
BOTTLENECK_CATEGORICAL_FEATURES = []
BOTTLENECK_TARGET = None
BOTTLENECK_FEATURES = BOTTLENECK_NUMERIC_FEATURES + BOTTLENECK_CATEGORICAL_FEATURES

# ---- 4.4 Project Performance / Health Analysis (Phase 2) ----
PERFORMANCE_NUMERIC_FEATURES = [
    "project_progress_percentage", "overdue_task_count", "completed_task_count",
    "blocked_task_count", "workload_ratio", "team_size", "historical_delay_rate",
    "progress_variance", "completion_velocity",
]
PERFORMANCE_CATEGORICAL_FEATURES = ["milestone_status"]
PERFORMANCE_TARGET = "health_score"  # continuous, 0-100 — must not be fabricated
PERFORMANCE_FEATURES = PERFORMANCE_NUMERIC_FEATURES + PERFORMANCE_CATEGORICAL_FEATURES

# ---- Registry used by the pipeline orchestrator (Section 8 of this notebook) ----
TASK_SCHEMAS = {
    TASK_DELAY_PREDICTION: {
        "numeric": DELAY_NUMERIC_FEATURES, "categorical": DELAY_CATEGORICAL_FEATURES,
        "features": DELAY_FEATURES, "target": DELAY_TARGET,
        "problem_type": "binary_classification", "leakage_fields": DELAY_LEAKAGE_FIELDS,
    },
    TASK_RISK_ANALYSIS: {
        "numeric": RISK_NUMERIC_FEATURES, "categorical": RISK_CATEGORICAL_FEATURES,
        "features": RISK_FEATURES, "target": RISK_TARGET,
        "problem_type": "multiclass_classification", "leakage_fields": [],
    },
    TASK_BOTTLENECK_DETECTION: {
        "numeric": BOTTLENECK_NUMERIC_FEATURES, "categorical": BOTTLENECK_CATEGORICAL_FEATURES,
        "features": BOTTLENECK_FEATURES, "target": BOTTLENECK_TARGET,  # None -> unsupervised
        "problem_type": "anomaly_detection", "leakage_fields": [],
    },
    TASK_PERFORMANCE_ANALYSIS: {
        "numeric": PERFORMANCE_NUMERIC_FEATURES, "categorical": PERFORMANCE_CATEGORICAL_FEATURES,
        "features": PERFORMANCE_FEATURES, "target": PERFORMANCE_TARGET,
        "problem_type": "regression", "leakage_fields": [],
    },
}
print("Feature/target schemas defined for:", list(TASK_SCHEMAS.keys()))

## 5. Preprocessing Steps

Applied identically across all four tasks:

1. **Load & deduplicate** — read the CSV dataset, drop exact duplicate rows.
2. **Column validation** — confirm all required feature and target columns
   exist before training starts, failing fast with a clear error if the
   data preparation task's output doesn't match.
3. **Leakage guard** — raise an error if `completed_date`, `actual_duration`,
   or `final_delay_days` end up in a feature list (Section 10 of the Model
   Approach document).
4. **Numeric preprocessing** — median imputation, then standard scaling.
5. **Categorical preprocessing** — constant `"missing"` imputation, then
   one-hot encoding (`handle_unknown="ignore"` so unseen categories at
   inference time don't break the pipeline).
6. **Train/test split** — stratified 80/20 split for the classification
   tasks, keeping class balance in both sets.

All of this is wrapped inside each model's `Pipeline`, so imputers, scalers,
and encoders are always fit only on training data (no data leakage).

In [ ]:
def load_dataset(file_path: str) -> pd.DataFrame:
    """Load the prepared dataset produced by the data preparation task."""
    df = pd.read_csv(file_path)
    df = df.drop_duplicates()
    return df


def basic_data_checks(df: pd.DataFrame, required_columns: list) -> None:
    """Confirm the dataset has every column a task needs before training."""
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(
            f"Dataset is missing required columns: {missing}. "
            "Confirm the data preparation task has produced these fields."
        )


def check_no_leakage(feature_list: list, leakage_fields: list) -> None:
    """Guard against data leakage (Model Approach, Section 10)."""
    leaked = [f for f in leakage_fields if f in feature_list]
    if leaked:
        raise ValueError(
            f"Data leakage detected: {leaked} must not be used as input "
            "features — they are only used to construct the target label."
        )


def build_preprocessor(numeric_features: list, categorical_features: list) -> ColumnTransformer:
    """Median-impute + scale numeric columns; impute + one-hot categorical columns."""
    transformers = []
    if numeric_features:
        numeric_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        transformers.append(("numeric", numeric_pipeline, numeric_features))
    if categorical_features:
        categorical_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("categorical", categorical_pipeline, categorical_features))
    return ColumnTransformer(transformers=transformers)


def split_features_target(df: pd.DataFrame, features: list, target: str = None):
    """Split into X (features) and y (target); y is None for Bottleneck Detection."""
    X = df[features].copy()
    y = df[target].copy() if target else None
    return X, y


def train_test_split_data(X, y, stratify: bool = True):
    """80/20 split, stratified by default for classification targets."""
    stratify_arg = y if stratify else None
    return train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify_arg)

print("Preprocessing utilities ready.")

## 6. Model Pipelines

| Task | Problem Type | Model | Main Output |
|---|---|---|---|
| Task Delay Prediction | Binary Classification | Random Forest Classifier | Delay probability + delayed/on-time label |
| Risk Analysis | Classification | Random Forest Classifier | Risk probability + risk level |
| Bottleneck Detection | Anomaly Detection | Isolation Forest | Bottleneck/anomaly score |
| Performance Analysis (Phase 2) | Regression | Random Forest Regressor | Performance/health score |

Tree-based models are the preferred starting point because ScaleFlow works
mainly with structured, tabular project data, and they stay interpretable —
which matters because the platform needs to explain *why* a task or project
was flagged, not just produce a score (Model Approach, Section 3).

### 6.1 Task Delay Prediction — Random Forest Classifier

Binary classification. `class_weight="balanced"` is used because delayed
tasks are typically the minority class, and recall on the delayed class
matters more than raw accuracy (Model Approach, Section 4).

In [ ]:
def build_delay_model() -> Pipeline:
    """preprocessing -> Random Forest Classifier (is_delayed: 1/0)"""
    preprocessor = build_preprocessor(DELAY_NUMERIC_FEATURES, DELAY_CATEGORICAL_FEATURES)
    classifier = RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )
    return Pipeline(steps=[("preprocessing", preprocessor), ("classifier", classifier)])

# Grid for later hyperparameter tuning
DELAY_HYPERPARAMETER_GRID = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [None, 8, 12, 20],
    "classifier__min_samples_leaf": [1, 2, 4],
}
print("Delay Prediction model builder ready.")

### 6.2 Risk Analysis — Random Forest Classifier

Classification of Low / Medium / High risk. A regression alternative is
included in case the dataset provides a continuous `risk_score` instead of
discrete risk classes, as the Model Approach document explicitly allows.

In [ ]:
def build_risk_model() -> Pipeline:
    """preprocessing -> Random Forest Classifier (risk_level: low/medium/high)"""
    preprocessor = build_preprocessor(RISK_NUMERIC_FEATURES, RISK_CATEGORICAL_FEATURES)
    classifier = RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )
    return Pipeline(steps=[("preprocessing", preprocessor), ("classifier", classifier)])


def build_risk_score_regressor() -> Pipeline:
    """Alternative pipeline if the target is continuous risk_score (0-100) instead."""
    preprocessor = build_preprocessor(RISK_NUMERIC_FEATURES, RISK_CATEGORICAL_FEATURES)
    regressor = RandomForestRegressor(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    return Pipeline(steps=[("preprocessing", preprocessor), ("regressor", regressor)])

RISK_HYPERPARAMETER_GRID = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [None, 8, 12, 20],
    "classifier__min_samples_leaf": [1, 2, 4],
}
print("Risk Analysis model builder ready.")

### 6.3 Bottleneck Detection — Isolation Forest

Unsupervised anomaly detection: no labeled bottleneck data exists yet, so
this model trains without a target column. `contamination` is a starting
assumption for the expected proportion of bottleneck-like rows, to be
refined once project managers validate the flagged cases (Model Approach,
Section 6).

In [ ]:
def build_bottleneck_model() -> Pipeline:
    """preprocessing -> Isolation Forest (unsupervised, no target column)"""
    preprocessor = build_preprocessor(BOTTLENECK_NUMERIC_FEATURES, BOTTLENECK_CATEGORICAL_FEATURES)
    detector = IsolationForest(
        n_estimators=300, contamination=BOTTLENECK_CONTAMINATION,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    return Pipeline(steps=[("preprocessing", preprocessor), ("detector", detector)])

BOTTLENECK_HYPERPARAMETER_GRID = {
    "detector__n_estimators": [150, 300, 500],
    "detector__contamination": [0.05, 0.1, 0.15],
}
print("Bottleneck Detection model builder ready.")

### 6.4 Project Performance / Health Analysis — Random Forest Regressor (Phase 2)

Deferred (Model Approach, Section 11): a clearly defined numerical health
target is required before this model can be trained, and that target must
not be fabricated. The pipeline is prepared here so training can start as
soon as historical project-health scores are available.

In [ ]:
def build_performance_model() -> Pipeline:
    """preprocessing -> Random Forest Regressor (health_score). Phase 2 — do not
    train until a real health_score target exists."""
    preprocessor = build_preprocessor(PERFORMANCE_NUMERIC_FEATURES, PERFORMANCE_CATEGORICAL_FEATURES)
    regressor = RandomForestRegressor(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    return Pipeline(steps=[("preprocessing", preprocessor), ("regressor", regressor)])

PERFORMANCE_HYPERPARAMETER_GRID = {
    "regressor__n_estimators": [200, 300, 500],
    "regressor__max_depth": [None, 8, 12, 20],
    "regressor__min_samples_leaf": [1, 2, 4],
}
print("Performance Analysis model builder ready (Phase 2 — do not run until health_score exists).")

## 7. Reusable Training Functions

Shared across all four ML problems. `train_model()` accepts `y_train=None`
for the unsupervised Bottleneck Detection pipeline.

In [ ]:
def train_model(pipeline, X_train, y_train=None):
    """Fit a pipeline; y_train is omitted for the unsupervised detector."""
    if y_train is None:
        pipeline.fit(X_train)
    else:
        pipeline.fit(X_train, y_train)
    return pipeline


def cross_validate_model(pipeline, X, y, scoring: str = "f1_weighted", cv: int = CV_FOLDS):
    """Stratified k-fold CV for classification tasks (Delay, Risk)."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_score(pipeline, X, y, cv=skf, scoring=scoring, n_jobs=-1)


def cross_validate_regressor(pipeline, X, y, scoring: str = "neg_mean_absolute_error", cv: int = CV_FOLDS):
    """K-fold CV for the Performance/Health regressor."""
    kf = KFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_score(pipeline, X, y, cv=kf, scoring=scoring, n_jobs=-1)


def tune_hyperparameters(pipeline, param_grid: dict, X_train, y_train,
                          scoring: str = "f1_weighted", cv: int = CV_FOLDS):
    """Grid search with stratified CV; returns the fitted GridSearchCV object."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(pipeline, param_grid, scoring=scoring, cv=skf, n_jobs=-1, verbose=1)
    search.fit(X_train, y_train)
    return search


def save_model(pipeline, task_name: str, filename: str = None) -> str:
    """Persist a trained pipeline to artifacts/models."""
    filename = filename or f"{task_name}_pipeline.joblib"
    file_path = os.path.join(MODELS_DIR, filename)
    joblib.dump(pipeline, file_path)
    return file_path


def load_model(file_path: str):
    """Load a previously trained pipeline from disk."""
    return joblib.load(file_path)

print("Training utilities ready.")

## 8. Reusable Evaluation Functions

Metrics match Sections 4, 5, 6, 7 of the Model Approach document exactly:

- **Delay Prediction:** Precision, Recall, F1, ROC-AUC, Accuracy (secondary)
- **Risk Analysis:** Precision, Recall, F1, ROC-AUC, Confusion Matrix
- **Bottleneck Detection:** anomaly score + manual validation (Precision/Recall
  once labeled outcomes exist — no bottleneck label is invented here)
- **Performance Analysis:** MAE, RMSE, R²

In [ ]:
def evaluate_classification(pipeline, X_test, y_test, average: str = "weighted") -> dict:
    """Used for Task Delay Prediction and Risk Analysis."""
    y_pred = pipeline.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),  # secondary metric
        "precision": precision_score(y_test, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_test, y_pred, average=average, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average=average, zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "classification_report": classification_report(y_test, y_pred, zero_division=0),
    }
    # ROC-AUC only applies cleanly to binary classification with predict_proba
    if hasattr(pipeline, "predict_proba") and len(set(y_test)) == 2:
        try:
            y_proba = pipeline.predict_proba(X_test)[:, 1]
            metrics["roc_auc"] = roc_auc_score(y_test, y_proba)
        except Exception:
            metrics["roc_auc"] = None
    return metrics


def evaluate_bottleneck_detection(pipeline, X) -> dict:
    """No labeled ground truth yet -> report anomaly scores + flagged rows,
    ready for manual validation by project managers."""
    raw_scores = pipeline.decision_function(X)      # higher = more normal
    predictions = pipeline.predict(X)                 # -1 = anomaly, 1 = normal
    is_bottleneck = predictions == -1
    metrics = {
        "num_records": int(len(X)),
        "num_flagged_bottlenecks": int(is_bottleneck.sum()),
        "flagged_ratio": float(is_bottleneck.mean()),
        "anomaly_score_mean": float(np.mean(raw_scores)),
        "anomaly_score_min": float(np.min(raw_scores)),
        "anomaly_score_max": float(np.max(raw_scores)),
        "note": ("No labeled bottleneck data exists yet. Evaluate by manual "
                 "validation of flagged rows; compute Precision/Recall once "
                 "labeled outcomes become available."),
    }
    return metrics, is_bottleneck, raw_scores


def evaluate_regression(pipeline, X_test, y_test) -> dict:
    """Used for the Phase 2 Project Performance / Health regressor."""
    y_pred = pipeline.predict(X_test)
    return {
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
        "r2": r2_score(y_test, y_pred),
    }


def save_evaluation_report(metrics: dict, task_name: str) -> str:
    """Save metrics as JSON (+ a separate text file for the classification report)."""
    file_path = os.path.join(REPORTS_DIR, f"{task_name}_evaluation.json")
    serializable_metrics = {k: v for k, v in metrics.items() if k != "classification_report"}
    with open(file_path, "w") as f:
        json.dump(serializable_metrics, f, indent=2)
    if "classification_report" in metrics:
        text_report_path = os.path.join(REPORTS_DIR, f"{task_name}_classification_report.txt")
        with open(text_report_path, "w") as f:
            f.write(metrics["classification_report"])
    return file_path


def print_summary(metrics: dict, task_name: str) -> None:
    """Short, readable console summary after each run."""
    print(f"\n=== Evaluation Summary: {task_name} ===")
    for key, value in metrics.items():
        if key in ("classification_report", "confusion_matrix", "note"):
            continue
        print(f"{key:>10}: {value:.4f}" if isinstance(value, float) else f"{key:>10}: {value}")

print("Evaluation utilities ready.")

## 9. Pipeline Orchestration

`run_task_pipeline()` runs one task end-to-end, branching by `problem_type`:

```
validate columns → check for leakage → split X/y → build model
   → (classification/regression: train/test split + CV)
   → (anomaly detection: fit on full X, no split)
   → train → evaluate → save model + report
```

`run_mvp_tasks()` runs the three Phase 1 tasks in sequence (Delay
Prediction, Risk Analysis, Bottleneck Detection), matching Section 11 of the
Model Approach document. Performance Analysis (Phase 2) is intentionally
excluded from this default run.

In [ ]:
MODEL_BUILDERS = {
    TASK_DELAY_PREDICTION: build_delay_model,
    TASK_RISK_ANALYSIS: build_risk_model,
    TASK_BOTTLENECK_DETECTION: build_bottleneck_model,
    TASK_PERFORMANCE_ANALYSIS: build_performance_model,
}


def run_task_pipeline(df, task_name: str, save_artifacts: bool = True) -> dict:
    """Run the full pipeline for a single task; behavior branches by problem_type."""
    if task_name not in TASK_SCHEMAS:
        raise ValueError(f"Unknown task '{task_name}'. Valid options: {list(TASK_SCHEMAS.keys())}")

    schema = TASK_SCHEMAS[task_name]
    target = schema["target"]
    required_columns = schema["features"] + ([target] if target else [])
    basic_data_checks(df, required_columns)
    check_no_leakage(schema["features"], schema.get("leakage_fields", []))

    X, y = split_features_target(df, schema["features"], target)
    pipeline = MODEL_BUILDERS[task_name]()

    # ---- Anomaly detection (Bottleneck Detection): unsupervised ----
    if schema["problem_type"] == "anomaly_detection":
        pipeline = train_model(pipeline, X)  # no y
        metrics, is_bottleneck, scores = evaluate_bottleneck_detection(pipeline, X)
        print_summary(metrics, task_name)
        if save_artifacts:
            print(f"[{task_name}] Model saved to: {save_model(pipeline, task_name)}")
            print(f"[{task_name}] Report saved to: {save_evaluation_report(metrics, task_name)}")
        return {"pipeline": pipeline, "metrics": metrics, "is_bottleneck": is_bottleneck, "scores": scores}

    # ---- Regression (Performance Analysis, Phase 2) ----
    if schema["problem_type"] == "regression":
        X_train, X_test, y_train, y_test = train_test_split_data(X, y, stratify=False)
        cv_scores = cross_validate_regressor(pipeline, X_train, y_train)
        print(f"[{task_name}] Cross-validation neg-MAE: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
        pipeline = train_model(pipeline, X_train, y_train)
        metrics = evaluate_regression(pipeline, X_test, y_test)
        print_summary(metrics, task_name)
        if save_artifacts:
            print(f"[{task_name}] Model saved to: {save_model(pipeline, task_name)}")
            print(f"[{task_name}] Report saved to: {save_evaluation_report(metrics, task_name)}")
        return {"pipeline": pipeline, "metrics": metrics, "cv_scores": cv_scores}

    # ---- Classification (Delay Prediction, Risk Analysis) ----
    X_train, X_test, y_train, y_test = train_test_split_data(X, y, stratify=True)
    cv_scores = cross_validate_model(pipeline, X_train, y_train)
    print(f"[{task_name}] Cross-validation F1 (weighted): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
    pipeline = train_model(pipeline, X_train, y_train)
    metrics = evaluate_classification(pipeline, X_test, y_test)
    print_summary(metrics, task_name)
    if save_artifacts:
        print(f"[{task_name}] Model saved to: {save_model(pipeline, task_name)}")
        print(f"[{task_name}] Report saved to: {save_evaluation_report(metrics, task_name)}")
    return {"pipeline": pipeline, "metrics": metrics, "cv_scores": cv_scores}


def run_mvp_tasks(df) -> dict:
    """Run the Phase 1 / MVP tasks only: Delay Prediction, Risk Analysis, Bottleneck Detection."""
    results = {}
    for task_name in MVP_TASKS:
        print(f"\n{'=' * 60}\nRunning pipeline for: {task_name}\n{'=' * 60}")
        results[task_name] = run_task_pipeline(df, task_name)
    return results

print("Pipeline orchestration ready.")

## 10. Ready to Receive the Prepared Dataset

This setup does not train on real data yet. Once the data preparation task
delivers the dataset, place it at `data/raw/scaleflow_tasks_dataset.csv`
(or update `RAW_DATASET_FILE` above) and run the cell below.

The cell only loads the dataset if the file is present, so this notebook
can be run safely before the real data arrives — it will simply report
that it is still waiting for it.

In [ ]:
if os.path.exists(RAW_DATASET_FILE):
    df = load_dataset(RAW_DATASET_FILE)
    print(f"Dataset loaded from: {RAW_DATASET_FILE}")
    print(f"Dataset shape: {df.shape}")
    display(df.head())
else:
    df = None
    print(f"Waiting for the prepared dataset at: {RAW_DATASET_FILE}")
    print("Place the CSV there (or update RAW_DATASET_FILE above) to continue.")

## 11. Run the Phase 1 / MVP Pipelines

Runs Task Delay Prediction, Risk Analysis, and Bottleneck Detection
end-to-end once `df` has been loaded above. This cell is skipped
automatically if the dataset has not arrived yet.

In [ ]:
if df is not None:
    results = run_mvp_tasks(df)
else:
    print("Skipping training — no dataset loaded yet. Run the cell above once "
          "the prepared dataset from the data preparation task is available.")

## 12. Phase 2 — Project Performance / Health Analysis

Do **not** run this until the dataset contains a real, non-fabricated
`health_score` column (Model Approach, Section 7 & 11). Uncomment the cell
below only once that target exists.

In [ ]:
# if df is not None and PERFORMANCE_TARGET in df.columns:
#     performance_results = run_task_pipeline(df, TASK_PERFORMANCE_ANALYSIS)
# else:
#     print("Performance Analysis is deferred: no health_score target available yet.")

## 13. Modeling Workflow — Documentation & Next Steps

**Workflow summary:**

```
ScaleFlow Database → Data Processing → Feature Engineering
   → Delay Model | Risk Model | Bottleneck Model | Performance Model
   → Predictions & Scores → AI/ML Intelligence Layer
   → Generative AI / Report Engine → Recommendations & Insights
```

This matches Section 9 of the Model Approach document: each model draws on
the same processed and engineered feature set, and all model outputs feed a
shared intelligence layer that the Generative AI service converts into
reports and recommendations.

**Next steps for training:**

- **Data validation pass:** once the real dataset arrives, run
  `basic_data_checks()` and `check_no_leakage()` first to confirm column
  names match and no leakage fields slipped into the feature list.
- **Exploratory Data Analysis (EDA):** check class balance (delayed vs.
  on-time, risk levels), missing-value rates, and feature distributions
  before full training.
- **Baseline training run:** run Section 11 above for a first pass on the
  three MVP tasks and establish a baseline to improve on.
- **Bottleneck validation loop:** have project managers manually review the
  flagged bottleneck cases from Isolation Forest; once enough validated
  outcomes accumulate, treat it as a labeled dataset and switch to
  Precision/Recall evaluation via `evaluate_classification()`.
- **Hyperparameter tuning:** use `tune_hyperparameters()` with the grids
  defined next to each model once baseline results are available.
- **Risk Analysis target check:** confirm whether `risk_level`
  (classification) or `risk_score` (regression) is the actual target being
  produced, and switch to `build_risk_score_regressor()` if it is the latter.
- **Performance Analysis (Phase 2):** do not train this model until a real
  `health_score` exists. Until then, the platform can calculate an initial
  health indicator directly from defined project KPIs (Model Approach,
  Section 7).
- **System integration:** once a model is finalized, wrap `load_model()` +
  `pipeline.predict()` inside the backend's AI/ML Engine service so the
  Generative AI layer can consume predictions in real time (Model Approach,
  Section 9; Project Proposal, Section 6).

**References**
- ScaleFlow — Project Analysis & Proposal. BinX Tech, 2026.
- ScaleFlow — AI/ML Model Approach. BinX Tech, Team 4, 14 September 2026.